# Notebook 06: Neural Collaborative Filtering

## Why This Matters

Matrix factorization predicts $\hat{r}_{ui} = \mathbf{p}_u \cdot \mathbf{q}_i$ — a dot product of learned embeddings. This is a powerful model, but the dot product is a severely constrained interaction function. It assumes the user-item relationship can be captured by a linear combination of latent factor products. In practice, user preferences are non-linear: a user who loves action movies and comedies might rate action-comedies particularly high — a non-linear interaction that dot product can't represent.

He et al. (2017) "Neural Collaborative Filtering" showed that replacing the dot product with an MLP consistently improves over pure MF on several benchmarks. This notebook implements the full NeuMF architecture and explores when the extra complexity actually helps.

## Background

### The Dot Product Bottleneck

Dot product imposes a **triangle inequality** on item similarities: if user $u$ likes items $i$ and $j$, and item $k$ is similar to $i$, then the dot product model will also score $k$ high for $j$. This transitivity is sometimes wrong — user preferences aren't metric spaces.

More formally, dot product satisfies:
$$\hat{r}_{ui} = \sum_f p_{uf} q_{if}$$

This is linear in each factor. The NCF insight: replace this with a learned, non-linear function $\hat{r}_{ui} = f(\mathbf{p}_u, \mathbf{q}_i | \Theta)$ where $f$ is a multi-layer perceptron.

### NCF Variants

**GMF (Generalized Matrix Factorization)**: Element-wise product of user and item embeddings, followed by a linear output layer. Generalizes standard MF — when the output weight is a uniform vector, it reduces to standard MF.

$$\hat{y}_{ui}^{GMF} = \mathbf{w}^T (\mathbf{p}_u^{GMF} \odot \mathbf{q}_i^{GMF})$$

**MLP**: Concatenate user and item embeddings, pass through MLP layers with ReLU activations.

$$\hat{y}_{ui}^{MLP} = \phi_L(\phi_{L-1}(\ldots \phi_1(\mathbf{p}_u^{MLP} \| \mathbf{q}_i^{MLP})))$$

**NeuMF (Neural Matrix Factorization)**: Combine GMF and MLP, concatenate their outputs, add a final sigmoid layer.

$$\hat{y}_{ui} = \sigma(\mathbf{h}^T [\mathbf{p}_u^{GMF} \odot \mathbf{q}_i^{GMF} \| \hat{y}_{ui}^{MLP}])$$

### Why Dot Product Still Wins in Retrieval

Despite NCF's accuracy gains, dot product dominates in retrieval systems because:
1. **ANN compatibility**: Only dot product and cosine similarity have efficient ANN index support (FAISS). MLP scores can't be pre-indexed.
2. **Speed**: Inference is $O(d)$ per item for dot product vs $O(d \cdot L)$ for MLP.
3. **Marginal gains**: At scale with good features and regularization, MLP gains shrink.

NeuMF's home is the **ranking stage** (Notebook 05), where you only score 100–1000 candidates and can afford the extra compute.

In [ ]:
%pip install -q torch numpy pandas matplotlib scikit-learn tqdm

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm
import os, requests, zipfile, io, warnings, time

warnings.filterwarnings("ignore")
np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.style.use("seaborn-v0_8-whitegrid")

# ── Load MovieLens 100K ──────────────────────────────────────────────────────
DATA_DIR = "/tmp/ml-100k"
if not os.path.exists(DATA_DIR):
    r = requests.get("https://files.grouplens.org/datasets/movielens/ml-100k.zip", timeout=60)
    zipfile.ZipFile(io.BytesIO(r.content)).extractall("/tmp/")

ratings = pd.read_csv(f"{DATA_DIR}/u.data", sep="\t",
                      names=["user_id", "item_id", "rating", "timestamp"])
ratings["user_idx"] = ratings.user_id - 1
ratings["item_idx"] = ratings.item_id - 1
n_users = ratings.user_idx.max() + 1
n_items = ratings.item_idx.max() + 1

# Temporal split
ratings = ratings.sort_values(["user_id", "timestamp"])
train_list, test_list = [], []
for uid, group in ratings.groupby("user_id"):
    n = len(group)
    cutoff = max(1, int(n * 0.8))
    train_list.append(group.iloc[:cutoff])
    test_list.append(group.iloc[cutoff:])
train_df = pd.concat(train_list).reset_index(drop=True)
test_df  = pd.concat(test_list).reset_index(drop=True)

user_pos_train = train_df.groupby("user_idx")["item_idx"].apply(set).to_dict()
user_pos_test  = test_df.groupby("user_idx")["item_idx"].apply(set).to_dict()

print(f"Users: {n_users}, Items: {n_items}")
print(f"Train: {len(train_df):,}, Test: {len(test_df):,}")
print(f"Device: {device}")
print("Ready.")

## 1. Point-wise Training Dataset

NCF can be trained pointwise (predict click probability per item) or pairwise (BPR). The original paper uses pointwise with binary cross-entropy — treat any interaction as positive (label=1), sample 4 random non-interacted items as negatives (label=0).

In [ ]:
class NCFDataset(Dataset):
    """
    Pointwise NCF dataset: (user, item, label).
    Positive: any rated item (label=1)
    Negative: n_neg randomly sampled unrated items per positive (label=0)
    """
    def __init__(self, train_df, n_items, user_pos_sets, n_neg=4):
        self.users = train_df.user_idx.values
        self.items = train_df.item_idx.values
        self.n_items = n_items
        self.user_pos = user_pos_sets
        self.n_neg = n_neg
        self._build()

    def _build(self):
        """Pre-build (user, item, label) triples."""
        user_list, item_list, label_list = [], [], []
        all_items = np.arange(self.n_items)

        for u, i in zip(self.users, self.items):
            user_list.append(u)
            item_list.append(i)
            label_list.append(1.0)

            pos_set = self.user_pos.get(u, set())
            for _ in range(self.n_neg):
                neg = np.random.randint(0, self.n_items)
                while neg in pos_set:
                    neg = np.random.randint(0, self.n_items)
                user_list.append(u)
                item_list.append(neg)
                label_list.append(0.0)

        self.X_users  = torch.tensor(user_list, dtype=torch.long)
        self.X_items  = torch.tensor(item_list, dtype=torch.long)
        self.y_labels = torch.tensor(label_list, dtype=torch.float32)

    def __len__(self):
        return len(self.y_labels)

    def __getitem__(self, idx):
        return self.X_users[idx], self.X_items[idx], self.y_labels[idx]


print("Building NCF dataset...")
ncf_ds = NCFDataset(train_df, n_items, user_pos_train, n_neg=4)
ncf_loader = DataLoader(ncf_ds, batch_size=1024, shuffle=True, num_workers=0)
print(f"Dataset size: {len(ncf_ds):,} (train + {4}x negatives) | {len(ncf_loader)} batches")

## 2. GMF, MLP, and NeuMF Models

We implement all three NCF variants with a common interface for easy comparison. NeuMF combines the GMF and MLP paths via a learned weighted combination.

In [ ]:
class GMF(nn.Module):
    """Generalized Matrix Factorization: element-wise product + linear output."""

    def __init__(self, n_users, n_items, n_factors=32):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, n_factors)
        self.item_emb = nn.Embedding(n_items, n_factors)
        self.out = nn.Linear(n_factors, 1)
        nn.init.normal_(self.user_emb.weight, 0, 0.01)
        nn.init.normal_(self.item_emb.weight, 0, 0.01)

    def forward(self, users, items):
        u = self.user_emb(users)
        i = self.item_emb(items)
        z = u * i  # element-wise product
        return torch.sigmoid(self.out(z)).squeeze(1)


class MLP_NCF(nn.Module):
    """MLP path: concatenate embeddings, pass through MLP."""

    def __init__(self, n_users, n_items, n_factors=32, layers=[64, 32, 16]):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, n_factors)
        self.item_emb = nn.Embedding(n_items, n_factors)
        nn.init.normal_(self.user_emb.weight, 0, 0.01)
        nn.init.normal_(self.item_emb.weight, 0, 0.01)

        mlp_layers = []
        in_size = 2 * n_factors
        for out_size in layers:
            mlp_layers.extend([nn.Linear(in_size, out_size), nn.ReLU(), nn.Dropout(0.2)])
            in_size = out_size
        mlp_layers.append(nn.Linear(in_size, 1))
        self.mlp = nn.Sequential(*mlp_layers)

    def forward(self, users, items):
        u = self.user_emb(users)
        i = self.item_emb(items)
        z = torch.cat([u, i], dim=1)
        return torch.sigmoid(self.mlp(z)).squeeze(1)


class NeuMF(nn.Module):
    """
    NeuMF: Combine GMF and MLP paths.
    GMF path: element-wise product
    MLP path: concatenation through MLP
    Combined: concatenate both paths → final sigmoid
    """

    def __init__(self, n_users, n_items, n_factors_gmf=16, n_factors_mlp=32,
                 mlp_layers=[64, 32, 16]):
        super().__init__()
        # GMF embeddings
        self.gmf_user = nn.Embedding(n_users, n_factors_gmf)
        self.gmf_item = nn.Embedding(n_items, n_factors_gmf)

        # MLP embeddings (separate from GMF for more expressivity)
        self.mlp_user = nn.Embedding(n_users, n_factors_mlp)
        self.mlp_item = nn.Embedding(n_items, n_factors_mlp)

        for emb in [self.gmf_user, self.gmf_item, self.mlp_user, self.mlp_item]:
            nn.init.normal_(emb.weight, 0, 0.01)

        # MLP
        mlp_in = 2 * n_factors_mlp
        mlp_seq = []
        for out_size in mlp_layers:
            mlp_seq.extend([nn.Linear(mlp_in, out_size), nn.ReLU(), nn.Dropout(0.2)])
            mlp_in = out_size
        self.mlp = nn.Sequential(*mlp_seq)

        # Output: concat GMF output + last MLP layer
        self.predict = nn.Linear(n_factors_gmf + mlp_layers[-1], 1)

    def forward(self, users, items):
        # GMF path
        gu = self.gmf_user(users)
        gi = self.gmf_item(items)
        gmf_out = gu * gi  # (B, n_factors_gmf)

        # MLP path
        mu = self.mlp_user(users)
        mi = self.mlp_item(items)
        mlp_in = torch.cat([mu, mi], dim=1)
        mlp_out = self.mlp(mlp_in)  # (B, mlp_layers[-1])

        # Combined
        combined = torch.cat([gmf_out, mlp_out], dim=1)
        return torch.sigmoid(self.predict(combined)).squeeze(1)


n_params_gmf  = sum(p.numel() for p in GMF(n_users, n_items).parameters())
n_params_mlp  = sum(p.numel() for p in MLP_NCF(n_users, n_items).parameters())
n_params_neumf = sum(p.numel() for p in NeuMF(n_users, n_items).parameters())
print(f"GMF parameters:   {n_params_gmf:>10,}")
print(f"MLP parameters:   {n_params_mlp:>10,}")
print(f"NeuMF parameters: {n_params_neumf:>10,}")

## 3. Training Loop

We train with binary cross-entropy loss (click prediction). The loss function is standard BCE — we're predicting the probability that a user will interact with an item.

In [ ]:
def train_ncf(model, loader, n_epochs=20, lr=1e-3, device="cpu", name="Model"):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
    criterion = nn.BCELoss()
    history = []

    for epoch in range(n_epochs):
        model.train()
        total_loss = 0
        for users, items, labels in loader:
            users, items, labels = users.to(device), items.to(device), labels.to(device)
            optimizer.zero_grad()
            preds = model(users, items)
            loss = criterion(preds, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        avg = total_loss / len(loader)
        history.append(avg)
        if (epoch + 1) % 5 == 0:
            print(f"  [{name}] Epoch {epoch+1:2d}/{n_epochs} | BCE Loss: {avg:.4f}")

    return history


def eval_hit_rate(model, user_pos_train, user_pos_test, n_items, K=10, n_eval=300, device="cpu"):
    """Hit Rate@K (HR@K): fraction of users where ≥1 test item is in top-K."""
    model.eval()
    hits = []
    eval_users = [u for u in user_pos_test if len(user_pos_test[u]) > 0][:n_eval]

    with torch.no_grad():
        for u in eval_users:
            test_pos = user_pos_test[u]
            seen = user_pos_train.get(u, set())

            all_users = torch.tensor([u] * n_items, dtype=torch.long).to(device)
            all_items = torch.arange(n_items, dtype=torch.long).to(device)
            scores = model(all_users, all_items).cpu().numpy()
            scores[list(seen)] = -np.inf

            top_k = np.argsort(scores)[::-1][:K]
            hits.append(1 if len(set(top_k) & test_pos) > 0 else 0)

    return np.mean(hits)


models_to_train = {
    "GMF":   GMF(n_users, n_items, n_factors=32),
    "MLP":   MLP_NCF(n_users, n_items, n_factors=32, layers=[64, 32, 16]),
    "NeuMF": NeuMF(n_users, n_items, n_factors_gmf=16, n_factors_mlp=32),
}

histories = {}
eval_results = {}
for name, m in models_to_train.items():
    print(f"\nTraining {name}...")
    t0 = time.time()
    histories[name] = train_ncf(m, ncf_loader, n_epochs=20, lr=1e-3, device=str(device), name=name)
    hr = eval_hit_rate(m, user_pos_train, user_pos_test, n_items, K=10, device=str(device))
    eval_results[name] = hr
    print(f"  HR@10: {hr:.4f} | Time: {time.time()-t0:.1f}s")

## 4. Comparing MF vs NeuMF

When does NeuMF actually beat standard matrix factorization? The answer depends on dataset characteristics: NeuMF tends to win when user preferences are genuinely non-linear — e.g., complex genre combinations, context-dependent preferences, or when users have diverse, multi-faceted taste profiles.

In [ ]:
# Add a standard MF (dot product) for baseline comparison
class MatrixFactorization(nn.Module):
    """Standard dot-product MF for comparison."""
    def __init__(self, n_users, n_items, n_factors=32):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, n_factors)
        self.item_emb = nn.Embedding(n_items, n_factors)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        nn.init.normal_(self.user_emb.weight, 0, 0.01)
        nn.init.normal_(self.item_emb.weight, 0, 0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, users, items):
        u = self.user_emb(users)
        i = self.item_emb(items)
        bu = self.user_bias(users).squeeze(1)
        bi = self.item_bias(items).squeeze(1)
        return torch.sigmoid((u * i).sum(dim=1) + bu + bi)


print("Training MF baseline...")
mf_model = MatrixFactorization(n_users, n_items, n_factors=32)
histories["MF (dot product)"] = train_ncf(mf_model, ncf_loader, n_epochs=20, device=str(device), name="MF")
hr_mf = eval_hit_rate(mf_model, user_pos_train, user_pos_test, n_items, K=10, device=str(device))
eval_results["MF (dot product)"] = hr_mf
print(f"  MF HR@10: {hr_mf:.4f}")

# Summary comparison
print("\n" + "="*50)
print(f"{'Model':<20} {'HR@10':>8} {'Parameters':>12}")
print("-" * 42)
all_models = {"MF (dot product)": mf_model, **models_to_train}
for name, m in all_models.items():
    n_p = sum(p.numel() for p in m.parameters())
    hr = eval_results.get(name, 0)
    print(f"{name:<20} {hr:>8.4f} {n_p:>12,}")

# Training curves
fig, ax = plt.subplots(figsize=(9, 4))
for name, hist in histories.items():
    ax.plot(hist, label=name)
ax.set_xlabel("Epoch")
ax.set_ylabel("BCE Loss")
ax.set_title("NCF Training Loss Curves")
ax.legend()
plt.tight_layout()
plt.savefig("/tmp/ncf_curves.png", dpi=120)
plt.show()

## 5. Embedding Space Visualization

A key difference between MF and NeuMF: in MF, the user and item embeddings must lie in a shared metric space compatible with dot product. NeuMF's separate embedding tables for GMF and MLP can represent different aspects of user and item identity.

In [ ]:
from sklearn.decomposition import PCA

# Compare embedding spaces: MF vs NeuMF GMF path
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
movies_meta = pd.read_csv(f"{DATA_DIR}/u.item", sep="|", encoding="latin-1", header=None,
                          usecols=[0,1,5,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23],
                          names=["item_id","title","Action","Comedy","Drama","Fantasy",
                                 "Horror","Musical","Mystery","Romance","Sci-Fi","Thriller",
                                 "War","Western","Animation","Children","Crime","Film-Noir","Documentary"])

for ax, (model_name, model_obj, emb_attr) in zip(axes, [
    ("MF Embeddings", mf_model, "item_emb"),
    ("NeuMF GMF Embeddings", models_to_train["NeuMF"], "gmf_item"),
]):
    emb_weights = getattr(model_obj, emb_attr).weight.detach().cpu().numpy()

    # PCA to 2D
    pca = PCA(n_components=2, random_state=42)
    emb_2d = pca.fit_transform(emb_weights)

    # Color by most popular genre
    genre_cols_vis = ["Action","Comedy","Drama","Romance","Sci-Fi","Horror"]
    colors_map = {"Action":"red","Comedy":"gold","Drama":"steelblue",
                  "Romance":"pink","Sci-Fi":"green","Horror":"purple","Other":"gray"}

    for item_id in range(min(500, n_items)):
        row = movies_meta[movies_meta.item_id == item_id + 1]
        if len(row) == 0:
            c = "gray"
        else:
            row = row.iloc[0]
            c = next((colors_map[g] for g in genre_cols_vis if row.get(g, 0) == 1), "gray")
        ax.scatter(emb_2d[item_id, 0], emb_2d[item_id, 1], c=c, alpha=0.4, s=15)

    # Legend
    for g, c in colors_map.items():
        ax.scatter([], [], c=c, label=g, s=30)
    ax.legend(fontsize=8, markerscale=1.5)
    ax.set_title(f"{model_name}\n(PCA 2D, colored by genre)")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

plt.tight_layout()
plt.savefig("/tmp/embedding_pca.png", dpi=120)
plt.show()
print("Genre clusters visible in embedding space confirm the model captures item semantics")

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Dot product bottleneck | Linear interaction — can't capture non-linear user preferences |
| GMF | Element-wise product + linear output — generalizes standard MF |
| MLP | Concatenation + MLP layers — learns non-linear interactions |
| NeuMF | Combines GMF + MLP paths — He et al. (2017) |
| Separate embeddings | GMF and MLP paths use separate embedding tables — more expressivity |
| BCE loss | Pointwise click probability — 4 negatives per positive is typical |
| When MLP helps | Complex, multi-faceted preferences; non-linear genre interactions |
| When dot product wins | Retrieval (ANN compatibility, speed); large catalogs |

### Common Pitfalls
- Using NeuMF for retrieval — MLP scores can't be indexed in FAISS
- Too many MLP layers with small datasets — overfits quickly
- Sharing embeddings between GMF and MLP paths — limits expressivity
- Not using dropout in MLP — overfits to popular users/items
- Ignoring that NeuMF gains are often marginal in practice — simple MF with good features often wins